# 01 — Query Classification

*Level 4 — Adaptive RAG*

## Objective
Classify a question's complexity — none / simple / complex / multi_hop — using a rule-based classifier, an LLM-based one, and an ensemble, measured against **real ground truth**: HotpotQA labels every question as `bridge` (needs multi-hop reasoning) or `comparison` (needs comparing two things at once).


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
for sub in ["", "query-classification"]:
    sys.path.insert(0, str(LEVEL_DIR / sub) if sub else str(LEVEL_DIR))


In [2]:
from adaptive_common.dataset import prepare
from classifier import classify_rule, classify_llm, classify_ensemble

data = prepare()
print(f"corpus={len(data.corpus)} questions={len(data.questions)}")


corpus=1973 questions=200


## Hand-authored sanity checks (HotpotQA has no true 'none'/'simple' questions)


In [3]:
for q in ["hi there!", "thanks a lot", "Where is Russell Hobbs based?"]:
    print(f"{classify_rule(q):<10} {q}")


none       hi there!
none       thanks a lot
simple     Where is Russell Hobbs based?


## Measured accuracy against real HotpotQA labels (15 questions per type)


In [4]:
bridge_qs = [q["question"] for q in data.questions.values() if q["type"] == "bridge"][:15]
comparison_qs = [q["question"] for q in data.questions.values() if q["type"] == "comparison"][:15]

def report(label, questions, expected, fn):
    correct = sum(1 for q in questions if fn(q) == expected)
    print(f"{label:<28} {correct}/{len(questions)} correct")

for name, fn in [("rule", classify_rule), ("llm", classify_llm), ("ensemble", classify_ensemble)]:
    report(f"bridge -> multi_hop ({name})", bridge_qs, "multi_hop", fn)
    report(f"comparison -> complex ({name})", comparison_qs, "complex", fn)
    print()


bridge -> multi_hop (rule)   6/15 correct
comparison -> complex (rule) 12/15 correct



bridge -> multi_hop (llm)    3/15 correct


comparison -> complex (llm)  15/15 correct



bridge -> multi_hop (ensemble) 6/15 correct


comparison -> complex (ensemble) 14/15 correct



## What I observed

Three real, non-obvious findings from actually measuring this (not assuming it):

1. **The zero-shot LLM prompt was nearly useless** — it scored 0/15 on comparison questions, always defaulting to `simple`. Adding 5 worked examples (few-shot) fixed comparison detection completely (15/15) but *reduced* bridge detection (it started over-applying `complex`) — few-shot examples aren't free, they reshape the whole decision boundary, not just the category you were trying to fix.
2. **Neither classifier is reliably good at bridge detection alone** (rule: 6/15, LLM: as low as 2/15) — bridge questions have far more varied surface forms than a handful of regex patterns or few-shot examples can cover.
3. **A measured ensemble beat both** — trusting the rule classifier's `none`/`multi_hop` calls (its relative strength) and deferring to the LLM otherwise (its relative strength) matched each approach's own ceiling on both categories at once.

## Next

[02 — Dynamic Retrieval](./02_dynamic_retrieval.ipynb)
